<a href="https://colab.research.google.com/github/A-ghori/sklearn/blob/main/Discretization_Binarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer

In [8]:
df = pd.read_csv('/content/Titanic-Dataset.csv',usecols=['Age','Fare','Survived'])
df.head(2)

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833


In [9]:
df.isnull().sum()

,0
Survived,0
Age,177
Fare,0


In [11]:
## drop the missing values rows
df.dropna(inplace=True)
#df.isnull().sum()

,0
Survived,0
Age,0
Fare,0


In [13]:
df.shape

(714, 3)

In [15]:
X = df.iloc[:,1:]
y = df.iloc[:,0]
y


,Survived
0,0
1,1
2,1
3,1
4,0
...,...
885,0
886,0
887,1
889,1


In [20]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)

In [21]:
print("Accuracy",accuracy)

Accuracy 0.6433566433566433


In [22]:
np.mean(cross_val_score(clf,X_train,y_train,scoring='accuracy'))

np.float64(0.6410068649885583)

## Now using K-Bins-discretizer strategy='uniform'

In [23]:
kbin_age = KBinsDiscretizer(n_bins=10, strategy='uniform', encode='ordinal')
kbin_fare = KBinsDiscretizer(n_bins=10, strategy='uniform',encode='ordinal')

In [32]:
trf = ColumnTransformer([
    ('first', kbin_age, ['Age']),
    ('second', kbin_fare, ['Fare'])
])

In [34]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)


In [36]:
## Inspect the trf
trf.named_transformers_['first']

KBinsDiscretizer(encode='ordinal', n_bins=10, strategy='uniform')

In [38]:
trf.named_transformers_['first'].n_bins_

array([10])

In [56]:
trf.named_transformers_['first'].bin_edges_[0]

array([ 0.42 ,  8.378, 16.336, 24.294, 32.252, 40.21 , 48.168, 56.126,
       64.084, 72.042, 80.   ])

In [41]:
trf.named_transformers_["second"].bin_edges_

array([array([  0.     ,  51.23292, 102.46584, 153.69876, 204.93168, 256.1646 ,
              307.39752, 358.63044, 409.86336, 461.09628, 512.3292 ])          ],
      dtype=object)

In [43]:
trf.named_transformers_['second'].get_params()

{'dtype': None,
 'encode': 'ordinal',
 'n_bins': 10,
 'random_state': None,
 'strategy': 'uniform',
 'subsample': 200000}

In [49]:
## Now form a DataFrame
output = pd.DataFrame({
    'age' : X_train['Age'],
    'age_trf' : X_train_trf[:,0],
    'fare' : X_train['Fare'],
    'fare_trf' : X_train_trf[:,1]
})

In [52]:
output['age_labels'] = pd.cut(x=X_train['Age'], bins=trf.named_transformers_['first'].bin_edges_[0].tolist())
output['fare_labels'] = pd.cut(x=X_train['Fare'], bins=trf.named_transformers_['second'].bin_edges_[0].tolist())

In [53]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_labels,fare_labels
348,3.0,0.0,15.9000,0.0,"(0.42, 8.378]","(0.0, 51.233]"
143,19.0,2.0,6.7500,0.0,"(16.336, 24.294]","(0.0, 51.233]"
171,4.0,0.0,29.1250,0.0,"(0.42, 8.378]","(0.0, 51.233]"
544,50.0,6.0,106.4250,2.0,"(48.168, 56.126]","(102.466, 153.699]"
177,50.0,6.0,28.7125,0.0,"(48.168, 56.126]","(0.0, 51.233]"


In [58]:
clf.fit(X_train_trf, y_train)
y_pred2 = clf.predict(X_test_trf)
print(accuracy_score(y_test,y_pred2))

0.6783216783216783


## Strategy='Quantile'

In [61]:
quantile = KBinsDiscretizer(n_bins=10, strategy='quantile',encode='ordinal')

In [63]:
trf2 = ColumnTransformer([
    ('Age',quantile,['Age']),
    ('Fare',quantile,['Fare'])
])

In [64]:
X_train_trf2 = trf2.fit_transform(X_train)
X_test_trf2 = trf2.transform(X_test)

In [66]:
clf.fit(X_train_trf2, y_train)
y_pred3 = clf.predict(X_test_trf2)
print(accuracy_score(y_test, y_pred3))

0.6223776223776224


In [68]:
trf2.named_transformers_

{'Age': KBinsDiscretizer(encode='ordinal', n_bins=10),
 'Fare': KBinsDiscretizer(encode='ordinal', n_bins=10)}

In [74]:
trf2.named_transformers_['Age'].bin_edges_[0]

array([ 0.42, 14.  , 19.  , 22.  , 25.  , 28.5 , 32.  , 36.  , 42.  ,
       50.  , 80.  ])

In [75]:
trf2.named_transformers_['Fare'].bin_edges_[0]

array([  0.    ,   7.75  ,   7.8958,   9.225 ,  13.    ,  15.75  ,
        26.    ,  29.125 ,  51.4792,  82.1708, 512.3292])

In [84]:
output = pd.DataFrame({
    'Age': X_train['Age'],
    'Age_trf2_':X_train_trf2[:,0],
    'Fare': X_train['Fare'],
    'Fare_trf2_':X_train_trf2[:,1]
})


In [86]:
#pd.cut(
#    x, -> Kis data ko bins mein divide karna hai
#    bins, -> Bin ki boundaries
#    labels=None, -> Har bin ko apna naam dena ho toh.
#    right=True, -> Ye decide karta hai bin ka right endpoint include hoga ya nahi.
#    include_lowest=False
#)

output['Age_labels'] = pd.cut(
    x=X_train['Age'],
    bins=trf2.named_transformers_['Age'].bin_edges_[0].tolist()
)
output['Fare_labels'] = pd.cut(
    x=X_train['Fare'],
    bins=trf2.named_transformers_['Fare'].bin_edges_[0].tolist()
)

In [87]:
output.sample(5)

,Age,Age_trf2_,Fare,Fare_trf2_,Age_labels,Fare_labels
765,51.0,9.0,77.9583,8.0,"(50.0, 80.0]","(51.479, 82.171]"
237,8.0,0.0,26.2500,6.0,"(0.42, 14.0]","(26.0, 29.125]"
120,21.0,2.0,73.5000,8.0,"(19.0, 22.0]","(51.479, 82.171]"
231,29.0,5.0,7.7750,1.0,"(28.5, 32.0]","(7.75, 7.896]"
251,29.0,5.0,10.4625,3.0,"(28.5, 32.0]","(9.225, 13.0]"


## Strategy = 'K-Means'

In [98]:
K_Means = KBinsDiscretizer(n_bins=10, strategy='kmeans',encode='ordinal')

In [99]:
trf3 = ColumnTransformer([
    ('Age',K_Means,['Age']),
    ('Fare',K_Means,['Fare'])
])


In [100]:
X_train_trf3 = trf3.fit_transform(X_train)
X_test_trf3 = trf3.transform(X_test)

In [101]:
clf.fit(X_train_trf3, y_train)
y_pred3 = clf.predict(X_test_trf3)
print(accuracy_score(y_test, y_pred3))

0.6153846153846154


In [102]:
output = pd.DataFrame({
    'Age': X_train['Age'],
    'Age_trf3_':X_train_trf2[:,0],
    'Fare': X_train['Fare'],
    'Fare_trf3_':X_train_trf2[:,1]
})


In [104]:
output['Age_Labels'] = pd.cut(
    x=X_train['Age'],
    bins=trf3.named_transformers_['Age'].bin_edges_[0].tolist()
    )
output['Fare_Labels'] = pd.cut(
    x=X_train['Fare'],
    bins=trf3.named_transformers_['Fare'].bin_edges_[0].tolist()
    )

In [105]:
output.sample(2)

,Age,Age_trf2_,Fare,Fare_trf2_,Age_Labels,Fare_Labels
167,45.0,8.0,27.9,6.0,"(40.345, 48.112]","(18.311, 38.121]"
820,52.0,9.0,93.5,9.0,"(48.112, 56.08]","(84.026, 107.124]"
